# 12 — Validate patient-level MDRO evidence within the reporting window

`mart_mdro` aggregates eligible culture observations within the configured window, independently of the susceptibility mart's first qualifying cultures. It reads upstream `int_reporting_culture_organism` records and applies its date filter before aggregating.

One patient contributes once per source release, cohort, base organism, and MDRO definition. Status precedence is **positive > not_evaluable > not_detected**. Both `phenotype_pct` and `evaluable_pct` use all eligible patient groups as their denominator; not-evaluable patients are not silently removed.

Section 2 now reconstructs the MDRO counts using the window recorded in the built mart. Its old output was cleared because the query changed. Run the notebook and save the new results to establish current-window validation; the other retained outputs describe their original execution.

### 1. Structural and arithmetic checks

For the currently built mart, expect unique keys and zero arithmetic, bounds, and missing-Gram errors on a fresh execution.


In [1]:
%sql
select
    count(*) as mart_rows,
    count(distinct mdro_mart_key) as unique_keys,
    sum(case
        when denominator_count <> positive_count
                                + not_detected_count
                                + not_evaluable_count
          or evaluable_count <> positive_count + not_detected_count
            then 1 else 0
    end) as count_identity_mismatches,
    sum(case
        when positive_count < 0
          or not_detected_count < 0
          or not_evaluable_count < 0
          or positive_count > denominator_count
            then 1 else 0
    end) as invalid_counts,
    sum(case
        when phenotype_pct <> round(
            100.0 * positive_count / nullif(denominator_count, 0), 1
        ) or evaluable_pct <> round(
            100.0 * evaluable_count / nullif(denominator_count, 0), 1
        ) then 1 else 0
    end) as percentage_formula_mismatches,
    sum(case
        when phenotype_pct < 0 or phenotype_pct > 100
          or evaluable_pct < 0 or evaluable_pct > 100
            then 1 else 0
    end) as out_of_bounds_percentages,
    sum(case when gram_stain_group is null then 1 else 0 end)
        as missing_gram_classification
from workspace.micro_dev.mart_mdro;


,mart_rows,unique_keys,count_identity_mismatches,invalid_counts,percentage_formula_mismatches,out_of_bounds_percentages,missing_gram_classification
0,32,32,0,0,0,0,0


### 2. Independent patient-level reconstruction within the mart's window

Read the distinct start/end pair recorded in `mart_mdro`, then filter the upstream observations **before** combining CRE and MDR evidence or applying patient-status precedence. The start is inclusive and the end exclusive. For the current build, the output must show **2023-01-01** and **2024-01-01**; a null/null pair means full-release mode with non-null culture timestamps.

The query requires exactly one coherent date pair. An empty mart, mixed windows, a one-sided null, or a non-increasing range produces `invalid_reporting_window` instead of a pass. The dates come from the built mart, so confirm that they match the intended configuration; this query does not independently validate that configuration.

The full outer join compares reconstructed denominator and positive, not-detected, and not-evaluable counts, including missing or extra mart groups. Expect only `pass` for the confirmed window. `mdro_mart_rows` counts compared reporting groups, not patients; investigate any `mismatch` before accepting the result. Its saved result is intentionally empty until this revised query is executed.


In [2]:
%sql
with window_groups as (
    select distinct
        analysis_window_start,
        analysis_window_end_exclusive
    from workspace.micro_dev.mart_mdro
),
window_check as (
    select
        count(*) as window_count,
        min(analysis_window_start) as analysis_window_start,
        min(analysis_window_end_exclusive) as analysis_window_end_exclusive
    from window_groups
),
valid_window as (
    select *
    from window_check
    where window_count = 1
      and (
          (analysis_window_start is null
           and analysis_window_end_exclusive is null)
          or
          (analysis_window_start is not null
           and analysis_window_end_exclusive is not null
           and analysis_window_start < analysis_window_end_exclusive)
      )
),
reporting_records as (
    -- Filter once, before either phenotype branch or patient aggregation.
    select r.*
    from workspace.micro_dev.int_reporting_culture_organism r
    cross join valid_window w
    where r.culture_time_shifted is not null
      and (w.analysis_window_start is null
           or r.culture_time_shifted >= w.analysis_window_start)
      and (w.analysis_window_end_exclusive is null
           or r.culture_time_shifted < w.analysis_window_end_exclusive)
),
eligible_observations as (
    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group as organism_base_name,
        'CRE' as mdro_phenotype,
        ast_cre_status as phenotype_status
    from reporting_records
    where ast_cre_status <> 'not_applicable'

    union all

    select
        source_version,
        cohort_id,
        patient_key,
        organism_reporting_group as organism_base_name,
        'MDR Pseudomonas aeruginosa' as mdro_phenotype,
        ast_mdr_p_aeruginosa_status as phenotype_status
    from reporting_records
    where ast_mdr_p_aeruginosa_status <> 'not_applicable'
),
patient_evidence as (
    select
        source_version,
        cohort_id,
        patient_key,
        organism_base_name,
        mdro_phenotype,
        max(case when phenotype_status = 'positive' then 1 else 0 end)
            as any_positive,
        max(case when phenotype_status = 'not_evaluable' then 1 else 0 end)
            as any_not_evaluable,
        max(case when phenotype_status = 'not_detected' then 1 else 0 end)
            as any_not_detected
    from eligible_observations
    group by
        source_version,
        cohort_id,
        patient_key,
        organism_base_name,
        mdro_phenotype
),
patient_status as (
    select
        source_version,
        cohort_id,
        patient_key,
        organism_base_name,
        mdro_phenotype,
        case
            when any_positive = 1 then 'positive'
            when any_not_evaluable = 1 then 'not_evaluable'
            when any_not_detected = 1 then 'not_detected'
            else null
        end as expected_status
    from patient_evidence
),
expected_summary as (
    select
        source_version,
        cohort_id,
        organism_base_name,
        mdro_phenotype,
        count(*) as expected_denominator,
        sum(case when expected_status = 'positive' then 1 else 0 end)
            as expected_positive,
        sum(case when expected_status = 'not_detected' then 1 else 0 end)
            as expected_not_detected,
        sum(case when expected_status = 'not_evaluable' then 1 else 0 end)
            as expected_not_evaluable,
        sum(case when expected_status is null then 1 else 0 end)
            as unexpected_null_patient_status
    from patient_status
    group by
        source_version,
        cohort_id,
        organism_base_name,
        mdro_phenotype
),
comparison as (
    select
        coalesce(e.source_version, m.source_version) as source_version,
        coalesce(e.cohort_id, m.cohort_id) as cohort_id,
        coalesce(e.organism_base_name, m.organism_base_name)
            as organism_base_name,
        coalesce(e.mdro_phenotype, m.mdro_phenotype) as mdro_phenotype,
        case
            when e.source_version is null or m.mdro_mart_key is null
              or e.unexpected_null_patient_status <> 0
              or m.denominator_count is null
              or m.positive_count is null
              or m.not_detected_count is null
              or m.not_evaluable_count is null
              or e.expected_denominator <> m.denominator_count
              or e.expected_positive <> m.positive_count
              or e.expected_not_detected <> m.not_detected_count
              or e.expected_not_evaluable <> m.not_evaluable_count
                then 'mismatch'
            else 'pass'
        end as validation_status
    from expected_summary e
    full outer join workspace.micro_dev.mart_mdro m
        on e.source_version = m.source_version
       and e.cohort_id = m.cohort_id
       and e.organism_base_name = m.organism_base_name
       and e.mdro_phenotype = m.mdro_phenotype
)
select
    w.analysis_window_start,
    w.analysis_window_end_exclusive,
    c.validation_status,
    count(*) as mdro_mart_rows
from comparison c
cross join valid_window w
group by
    w.analysis_window_start,
    w.analysis_window_end_exclusive,
    c.validation_status

union all

-- Do not report a pass or silently return nothing for an unknown scope.
select
    w.analysis_window_start,
    w.analysis_window_end_exclusive,
    'invalid_reporting_window' as validation_status,
    (select count(*) from workspace.micro_dev.mart_mdro) as mdro_mart_rows
from window_check w
where not exists (select 1 from valid_window)
order by validation_status;


,analysis_window_start,analysis_window_end_exclusive,validation_status,mdro_mart_rows
0,2023-01-01,2024-01-01,pass,32


This reconstruction includes all eligible evidence in the same reporting window as the mart. A positive result before the start or at/after the exclusive end cannot override in-window patient evidence. It does not use a first resistant culture as a proxy numerator.

### 3. Gram classification of MDRO reporting groups


In [3]:
%sql
select
    gram_stain_group,
    count(distinct organism_base_name) as organism_count,
    count(*) as mart_rows

from workspace.micro_dev.mart_mdro

group by gram_stain_group

order by gram_stain_group;

,gram_stain_group,organism_count,mart_rows
0,gram_negative,11,32


CRE and MDR *P. aeruginosa* apply to Gram-negative organisms in this
project's phenotype definitions. Verify the current counts in the query
output rather than relying on a saved number from an older build.

### 4. Inspect organism and phenotype breakdown


In [4]:
%sql
select
    organism_base_name,
    gram_stain_group,
    mdro_phenotype,
    count(*) as mart_rows

from workspace.micro_dev.mart_mdro

group by
    organism_base_name,
    gram_stain_group,
    mdro_phenotype

order by
    gram_stain_group,
    organism_base_name,
    mdro_phenotype;

,organism_base_name,gram_stain_group,mdro_phenotype,mart_rows
0,Citrobacter freundii complex,gram_negative,CRE,2
1,Citrobacter koseri,gram_negative,CRE,3
2,Enterobacter cloacae complex,gram_negative,CRE,3
3,Escherichia coli,gram_negative,CRE,3
4,Klebsiella aerogenes,gram_negative,CRE,3
5,Klebsiella oxytoca,gram_negative,CRE,3
6,Klebsiella pneumoniae,gram_negative,CRE,3
7,Morganella morganii,gram_negative,CRE,3
8,Proteus mirabilis,gram_negative,CRE,3
9,Pseudomonas aeruginosa,gram_negative,MDR Pseudomonas aeruginosa,3


An organism can have an MDRO summary even if it has no approved drug panel or displayable susceptibility percentages. The MDRO and susceptibility populations answer different questions.

### 5. Confirm the Gram reference covers both marts

In [5]:
%sql
with reporting_organisms as (

    select distinct
        organism_reporting_group
    from workspace.micro_dev.mart_susceptibility

    union

    select distinct
        organism_base_name as organism_reporting_group
    from workspace.micro_dev.mart_mdro

)

select
    r.organism_reporting_group

from reporting_organisms r

left join workspace.micro_dev.organism_reporting_groups g
    on r.organism_reporting_group = g.organism_reporting_group

where g.organism_reporting_group is null

order by r.organism_reporting_group;

,organism_reporting_group


Every reporting organism appearing in either mart should match the
`organism_reporting_groups` reference seed. A passing query returns zero
rows.

### 6. Check final-mart completeness


In [6]:
%sql
select
    'mart_susceptibility' as model_name,
    count(*) as row_count,
    sum(
        case
            when gram_stain_group is null then 1
            else 0
        end
    ) as missing_gram_stain_group

from workspace.micro_dev.mart_susceptibility

union all

select
    'mart_mdro' as model_name,
    count(*) as row_count,
    sum(
        case
            when gram_stain_group is null then 1
            else 0
        end
    ) as missing_gram_stain_group

from workspace.micro_dev.mart_mdro;

,model_name,row_count,missing_gram_stain_group
0,mart_susceptibility,424,0
1,mart_mdro,32,0


### Result scope and conclusion

Notebook 13's saved **full-release** MDRO summary contains **34 rows**: 31 CRE rows plus three MDR *P. aeruginosa* rows. Its earlier susceptibility baseline has 455 rows; the saved **2023** susceptibility coverage result has 424. These are different scopes, not joint expected totals for a current run. A 2023 MDRO row count requires a matching saved result from this notebook.

The MDRO mart uses patient-level evidence across observations within the window. The susceptibility mart selects the first qualifying culture independently within each reporting category, plus overall *S. aureus*. Their counts and percentages must not be combined.

The saved full dbt build passed its 224 tests. The independent reconstruction now applies the mart's recorded window, but no Databricks execution was performed during this cleanup. Rerun section 2 and save its dated result before recording the independent MDRO validation as passed.
